In [1]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

IMG_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\images"
LBL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\labels"


Device: cpu


In [2]:
class MRIDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, file_list):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.files = file_list

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img = cv2.imread(os.path.join(self.img_dir, img_name), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (224, 224))
        img = img.astype(np.float32) / 255.0
        img = np.stack([img, img, img], axis=0)
        img = torch.tensor(img, dtype=torch.float32)

        with open(os.path.join(self.lbl_dir, img_name.replace(".png", ".txt"))) as f:
            label = int(f.read().strip())

        return img, label


In [3]:
all_files = sorted(os.listdir(IMG_DIR))

dataset = MRIDataset(IMG_DIR, LBL_DIR, all_files)
loader = DataLoader(dataset, batch_size=8, shuffle=False)

print("Total samples:", len(dataset))


Total samples: 6939


In [4]:
cnn_model = timm.create_model(
    "resnet18",
    pretrained=True,
    num_classes=0
).to(device)

vit_model = timm.create_model(
    "vit_small_patch16_224",
    pretrained=True,
    num_classes=0
).to(device)

cnn_model.eval()
vit_model.eval()


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [5]:
all_features = []
all_labels = []

with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(device)

        cnn_feat = cnn_model(imgs)
        vit_feat = vit_model(imgs)

        combined = torch.cat([cnn_feat, vit_feat], dim=1)

        all_features.append(combined.cpu().numpy())
        all_labels.append(labels.numpy())

features = np.vstack(all_features)
labels = np.hstack(all_labels)

print("Feature shape:", features.shape)
print("Labels shape:", labels.shape)


Feature shape: (6939, 896)
Labels shape: (6939,)


In [6]:
MODEL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"

np.save(f"{MODEL_DIR}/cnnvit_features.npy", features)
np.save(f"{MODEL_DIR}/cnnvit_labels.npy", labels)

print("CNN+ViT features saved ")


CNN+ViT features saved 
